# Лабораторная работа 1 — рабочий ноутбук

**ФИО:**
**Файлы данных:**

Этот ноутбук — чистовой рабочий документ, а не черновик. Он должен
исполняться сверху вниз без ошибок на выданном вам **сыром** файле
и на выходе давать очищенную таблицу и `cleaning_log.csv`.

Разделы ниже обязательны, порядок менять не нужно. Внутри раздела
добавляйте столько ячеек, сколько нужно. Если какой-то раздел у вас
пустой — напишите в нём одной строкой, почему.

## 0. Окружение и настройки

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

RAW_CLIENTS = 'data/S15/clients.csv'
RAW_TXN = 'data/S15/transactions.csv'

## 1. Загрузка сырых данных

Файлы выгружены разными системами. Убедитесь, что они прочитались
**правильно**: кодировка, разделитель, десятичный разделитель, типы,
даты, идентификаторы.

In [2]:
raw_clients = pd.read_csv(
    RAW_CLIENTS,
    sep=',',
    decimal='.'
)

raw_transactions = pd.read_csv(
    RAW_TXN,
    sep=';',
    encoding='cp1251',
    decimal=','
)

display(raw_clients.head())
display(raw_transactions.head())

,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
0,C001919,1968-10-29,Ж,Санкт-Петербург,Массовый,Базовый,Дебетовая,14.06.2011,13.02.2016,Зарплата в нашем банке,53500.0,Да,2024-10,NaN,100,Да
1,C002920,1954-10-07,М,СПб,Массовый,Золото,Кредитная,02.04.2012,16.10.2016,Зарплата в другом банке,125500.0,Да,2024-03,2.0,120,Да
2,C000925,1973-02-04,М,г. Москва,Зарплатный,Золото,Дебетовая,18.04.2014,27.05.2016,Зарплата в нашем банке,69700.0,Нет,NaN,NaN,236,Да
3,C003287,1976-05-07,М,Нижний Новгород,Зарплатный,Серебро,Кредитная,09.06.2014,12.07.2019,Зарплата в нашем банке,38300.0,Нет,NaN,NaN,538,Да
4,C002354,1974-09-20,Ж,Москва,Массовый,Базовый,Дебетовая,29.09.2013,31.05.2018,Зарплата в другом банке,44.1,Да,2025-05,NaN,424,Да


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990124,C000052,2023-11-26 15:27:11,5482.85,RUB,Одежда и обувь,5651,Карта,успешно
1,990022,C002010,2023-12-10 11:38:58,1606.39,RUB,Аптеки,5912,СБП,успешно
2,100001,C003676,2024-01-01 00:54:00,384.01,RUB,Транспорт,4111,СБП,успешно
3,100002,C001643,2024-01-01 01:10:47,1769.76,RUB,Продукты,5411,Токен,успешно
4,100003,C004665,2024-01-01 02:06:49,58.73,RUB,Транспорт,4111,Карта,успешно


## 2. Аудит качества

Здесь вы ничего не исправляете — только находите и измеряете.
Для каждой метрики качества заранее определите знаменатель.

### 2.1. Объём, типы, ключи, единица наблюдения

In [5]:
audit_summary = pd.DataFrame([
    {
        'Таблица': 'clients',
        'Строк (N)': len(raw_clients),
        'Колонок': raw_clients.shape[1],
        'Ключ': 'client_id',
        'Уник. ключей (U)': raw_clients['client_id'].nunique(),
        'Дублей ключа': len(raw_clients) - raw_clients['client_id'].nunique(),
        'Доля брака ключа': f"{(len(raw_clients) - raw_clients['client_id'].nunique()) / len(raw_clients):.2%}"
    },
    {
        'Таблица': 'transactions',
        'Строк (N)': len(raw_transactions),
        'Колонок': raw_transactions.shape[1],
        'Ключ': 'txn_id',
        'Уник. ключей (U)': raw_transactions['txn_id'].nunique(),
        'Дублей ключа': len(raw_transactions) - raw_transactions['txn_id'].nunique(),
        'Доля брака ключа': f"{(len(raw_transactions) - raw_transactions['txn_id'].nunique()) / len(raw_transactions):.2%}"
    }
])

display(audit_summary)

,Таблица,Строк (N),Колонок,Ключ,Уник. ключей (U),Дублей ключа,Доля брака ключа
0,clients,5670,16,client_id,5615,55,0.97%
1,transactions,619916,9,txn_id,618416,1500,0.24%


### 2.2. Дубли

In [8]:
n_c = len(raw_clients)
full_dup_c = raw_clients.duplicated().sum()
key_dup_c = raw_clients.duplicated(subset=['client_id']).sum()
conflict_c = key_dup_c - full_dup_c

n_t = len(raw_transactions)
full_dup_t = raw_transactions.duplicated().sum()
key_dup_t = raw_transactions.duplicated(subset=['txn_id']).sum()
conflict_t = key_dup_t - full_dup_t

dup_summary = pd.DataFrame([
    {
        'Таблица': 'clients',
        'Строк (N)': n_c,
        'Ключ': 'client_id',
        'Полные клоны': full_dup_c,
        'Доля клонов': f"{full_dup_c / n_c:.2%}",
        'Конфликты ключа (неполные)': conflict_c,
        'Доля конфликтов': f"{conflict_c / n_c:.2%}"
    },
    {
        'Таблица': 'transactions',
        'Строк (N)': n_t,
        'Ключ': 'txn_id',
        'Полные клоны': full_dup_t,
        'Доля клонов': f"{full_dup_t / n_t:.2%}",
        'Конфликты ключа (неполные)': conflict_t,
        'Доля конфликтов': f"{conflict_t / n_t:.2%}"
    }
])

display(dup_summary)

print("\n--- Опасные конфликты данных в clients (одинаковый ID, разные данные) ---")
client_conflicts = raw_clients[raw_clients.duplicated(subset=['client_id'], keep=False) & ~raw_clients.duplicated(keep=False)]
display(client_conflicts.sort_values('client_id').head(6))

print("\n--- Опасные конфликты данных в transactions (одинаковый txn_id, разные данные) ---")
tx_conflicts = raw_transactions[raw_transactions.duplicated(subset=['txn_id'], keep=False) & ~raw_transactions.duplicated(keep=False)]
display(tx_conflicts.sort_values('txn_id').head(6))

,Таблица,Строк (N),Ключ,Полные клоны,Доля клонов,Конфликты ключа (неполные),Доля конфликтов
0,clients,5670,client_id,40,0.71%,15,0.26%
1,transactions,619916,txn_id,1266,0.20%,234,0.04%



--- Опасные конфликты данных в clients (одинаковый ID, разные данные) ---


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
478,C000221,1956-01-05,М,Казань,Массовый,Базовый,Дебетовая,20.09.2015,07.07.2018,Зарплата в другом банке,42200.0,Нет,NaN,2.0,445,Нет
4400,C000221,1956-01-05,М,Санкт-Петербург,Премиальный,Базовый,Дебетовая,20.09.2015,07.07.2018,Зарплата в другом банке,42200.0,Нет,NaN,2.0,445,Нет
854,C000328,1970-09-15,Ж,Казань,Массовый,Серебро,Дебетовая,29.01.2018,22.10.2020,Зарплата в нашем банке,108600.0,Да,2025-02,NaN,873,Нет
2063,C000574,1943-08-08,М,москва,Премиальный,Базовый,Дебетовая,04.10.2019,06.11.2019,Не указан,82300.0,Нет,NaN,NaN,244,Да
3576,C000574,1943-08-08,М,Новосибирск,Премиальный,Базовый,Дебетовая,04.10.2019,06.11.2019,Не указан,82300.0,Нет,NaN,NaN,244,Да
1705,C001470,1966-01-27,М,Санкт-Петербург,Зарплатный,Базовый,Дебетовая,31.08.2021,08.04.2023,Зарплата в нашем банке,118300.0,Да,2024-05,NaN,748,Да



--- Опасные конфликты данных в transactions (одинаковый txn_id, разные данные) ---


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
2787,102780,C005545,2024-01-05 05:51:33,561.09,RUB,Кафе и рестораны,5812,Карта,успешно
2788,102780,C005545,2024-01-05,561.09,RUB,Кафе и рестораны,5812,Карта,успешно
6956,106938,C005104,2024-01-08 11:57:04,9321.67,RUB,кафе и рестораны,5812,Онлайн-банк,успешно
6957,106938,C005104,2024-01-08,9321.67,RUB,кафе и рестораны,5812,Онлайн-банк,успешно
9546,109521,C001309,2024-01-11 19:51:49,1324.05,RUB,Аптеки,5912,Карта,успешно
9547,109521,C001309,2024-01-11,1324.05,RUB,Аптеки,5912,Карта,успешно


### 2.3. Соответствие данных описанию полей из задания

In [ ]:
missing_records = []
for name, df in [('clients', raw_clients), ('transactions', raw_transactions)]:
    null_counts = df.isna().sum()
    for col in df.columns:
        if null_counts[col] > 0:
            missing_records.append({
                'Таблица': name,
                'Колонка': col,
                'Пропусков (шт)': null_counts[col],
                'Доля от N': f"{null_counts[col] / len(df):.2%}"
            })

print("1. ПРОПУСКИ В ДАННЫХ")
display(pd.DataFrame(missing_records))


print("\n2. ЧИСЛОВЫЕ МЕТРИКИ")
num_cols = [
    ('clients', 'monthly_income', raw_clients['monthly_income']),
    ('clients', 'satisfaction_score', raw_clients['satisfaction_score']),
    ('transactions', 'amount', raw_transactions['amount'])
]

num_records = []
for tbl, col_name, s in num_cols:
    num_records.append({
        'Поле': f"{tbl}.{col_name}",
        'Min': s.min(),
        'Max': s.max(),
        'Значений < 0': (s < 0).sum(),
        'Значений == 0': (s == 0).sum()
    })

display(pd.DataFrame(num_records))


print("\n3. КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ")
cat_cols = [
    ('clients.segment', 4, raw_clients['segment']),
    ('clients.loyalty_tier', 4, raw_clients['loyalty_tier']),
    ('clients.card_type', 3, raw_clients['card_type']),
    ('clients.gender', 2, raw_clients['gender']),
    ('clients.has_mobile_app', 2, raw_clients['has_mobile_app']),
    ('clients.city', 'Города РФ', raw_clients['city']),
    ('transactions.category', 9, raw_transactions['category']),
    ('transactions.channel', 4, raw_transactions['channel']),
    ('transactions.status', 3, raw_transactions['status']),
    ('transactions.currency', 'Валюта', raw_transactions['currency']),
]

cat_records = []
for field, expected, s in cat_cols:
    unique_vals = sorted([str(x) for x in s.dropna().unique()])
    cat_records.append({
        'Поле': field,
        'Ожидалось по ТЗ': expected,
        'Факт (уник.)': len(unique_vals),
        'Все фактические значения': ", ".join(unique_vals)
    })

print("\n3. КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ (ПОЛНЫЙ СПИСОК ЗНАЧЕНИЙ)")
display(pd.DataFrame(cat_records))

1. ПРОПУСКИ В ДАННЫХ


,Таблица,Колонка,Пропусков (шт),Доля от N
0,clients,monthly_income,532,9.38%
1,clients,app_last_login,2103,37.09%
2,clients,satisfaction_score,3359,59.24%



2. ЧИСЛОВЫЕ МЕТРИКИ


,Поле,Min,Max,Значений < 0,Значений == 0
0,clients.monthly_income,-999.00,1052900.00,56,0
1,clients.satisfaction_score,1.00,5.00,0,0
2,transactions.amount,-60058.31,152959.72,7285,0



3. КАТЕГОРИАЛЬНЫЕ ПРИЗНАКИ


,Поле,Ожидалось по ТЗ,Факт (уник.),Значения в данных
0,clients.segment,4,4,"['Массовый', 'Зарплатный', 'Премиальный', 'Pri..."
1,clients.loyalty_tier,4,4,"['Базовый', 'Золото', 'Серебро', 'Платина']"
2,clients.card_type,3,3,"['Дебетовая', 'Кредитная', 'Премиальная']"
3,clients.gender,2,2,"['Ж', 'М']"
4,clients.has_mobile_app,2,2,"['Да', 'Нет']"
5,clients.city,Города РФ,17,"['Санкт-Петербург', 'СПб', 'г. Москва', 'Нижни..."
6,transactions.category,9,12,"['Одежда и обувь', 'Аптеки', 'Транспорт', 'Про..."
7,transactions.channel,4,4,"['Карта', 'СБП', 'Токен', 'Онлайн-банк']"
8,transactions.status,3,3,"['успешно', 'отклонено', 'возврат']"
9,transactions.currency,Валюта,3,"['RUB', 'EUR', 'USD']"


### 2.4. Связь между таблицами

**Выводы по аудиту.** Перечислите найденное: что это, сколько строк,
насколько это мешает решить задачу.

## 3. Препроцессинг

На каждое изменение — строка в журнале (раздел 4). Решение «оставить
как есть» тоже является решением и тоже попадает в журнал.

### 3.1. Типы, даты, числа

### 3.2. Категориальные признаки

### 3.3. Пропуски

Для каждого поля с пропусками: доля, предположение о механизме, решение.

### 3.4. Выбросы

Обнаружение и решение — это два разных шага. Сначала первый, потом второй.

### 3.5. Фильтры, применяемые перед расчётом активности

Покажите воронку: сколько строк на входе и сколько осталось после каждого фильтра.

## 4. Журнал изменений

Одна строка — одно решение, а не одна запись. Колонки: `field`, `issue`,
`scope`, `n_rows`, `value_before`, `action`, `value_after`, `rationale`.

In [ ]:
cleaning_log = pd.DataFrame(columns=[
    "field", "issue", "scope", "n_rows",
    "value_before", "action", "value_after", "rationale",
])

## 5. Разведочный анализ

Графиков столько, сколько нужно. В отчёт пойдут не все.

### 5.1. Распределения количественных признаков

### 5.2. Частотные характеристики категориальных признаков

### 5.3. Динамика по месяцам

### 5.4. Разрезы и связи между признаками

## 6. Метрика активности

Что именно вы считаете активностью, как она вычисляется, на каком
множестве клиентов, за какие окна. Отдельно: какое снижение эта метрика
**не увидит**.

## 7. Поиск и описание групп

Правило отбора для каждой группы, её размер, причина падения.
Правило разрешения конфликтов, если клиент подходит под несколько групп.

### 7.1. Кого в кампанию не берём и почему

## 8. Что проверили и что не подтвердилось

Гипотезы, которые вы проверили и отвергли. По одной строке на гипотезу,
с числом.

## 9. Сохранение результатов

Три файла: очищенная таблица для второго ноутбука, журнал изменений,
итоговый список кампании.

In [ ]:
# clean.to_parquet("clean_data.parquet", index=False)
# cleaning_log.to_csv("cleaning_log.csv", index=False, encoding="utf-8")
# campaign[["client_id", "group"]].to_csv("campaign_list.csv", index=False, encoding="utf-8")